# Khám Phá Dữ Liệu

Notebook này dùng để phân tích dữ liệu CAMELS của ngân hàng đã được làm sạch từ BigQuery hoặc từ các file CSV đã xử lý trong `data/processed/`.

Mục tiêu của notebook:
- Kiểm tra độ thiếu dữ liệu và phạm vi giá trị của các biến
- Quan sát phân phối của các chỉ số CAMELS
- Xem mối tương quan giữa các biến tài chính quan trọng
- Ghi nhận sớm các vấn đề dữ liệu cần xử lý trước khi làm ML

## 1. Chuẩn Bị

Notebook này đọc dữ liệu đã làm sạch từ `data/processed/`, nên có thể chạy cục bộ mà không cần BigQuery.
Điều này giúp bạn xem kết quả EDA ngay trước khi đưa dữ liệu lên warehouse.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.logger import get_logger

logger = get_logger(__name__)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["axes.titleweight"] = "bold"

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
fact_path = processed_dir / "fact_bank_performance_clean.csv"
dim_bank_path = processed_dir / "dim_bank_clean.csv"

fact_df = pd.read_csv(fact_path)
dim_bank_df = pd.read_csv(dim_bank_path)

df = fact_df.merge(
    dim_bank_df[["bank_key", "bank_code", "bank_name", "bank_type"]],
    on="bank_key",
    how="left",
)

logger.info("Đã tải %d dòng và %d cột từ các file CSV đã xử lý.", len(df), len(df.columns))
df.head()

## 2. Tổng Quan Dữ Liệu

Các cell tiếp theo sẽ tóm tắt kích thước dữ liệu, kiểu dữ liệu, độ thiếu và các thống kê cơ bản.

In [ ]:
display(df.info())
display(df.describe(include="all").T)

summary = pd.DataFrame({
    "cột": df.columns,
    "số_lượng_thiếu": df.isna().sum().values,
    "tỷ_lệ_thiếu_%": (df.isna().mean() * 100).round(2).values,
})
summary.sort_values("số_lượng_thiếu", ascending=False).reset_index(drop=True)

In [ ]:
numeric_cols = [
    "total_assets", "total_deposits", "total_loans", "total_equity",
    "npl_amount", "loan_loss_provision", "interest_income", "interest_expense",
    "net_interest_income", "profit_before_tax", "profit_after_tax",
    "npl_ratio", "llp_ratio", "roa", "roe", "nim", "cir", "eta", "etd", "lta", "ltd", "gta"
]
available_numeric_cols = [col for col in numeric_cols if col in df.columns]

fig, axes = plt.subplots(2, 1, figsize=(16, 10), constrained_layout=True)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0].head(20).plot(kind="bar", ax=axes[0], color="#c44e52")
axes[0].set_title("Top 20 cột có tỷ lệ thiếu cao nhất")
axes[0].set_ylabel("Tỷ lệ thiếu (%)")
axes[0].set_xlabel("Cột")
axes[0].tick_params(axis="x", rotation=75)

corr = df[available_numeric_cols].corr(numeric_only=True)
sns.heatmap(corr, cmap="RdBu_r", center=0, ax=axes[1], cbar_kws={"shrink": 0.8})
axes[1].set_title("Ma trận tương quan của các biến CAMELS")
plt.show()

In [ ]:
target_cols = ["npl_ratio", "llp_ratio", "roa", "roe", "nim", "cir", "eta", "etd"]
target_cols = [col for col in target_cols if col in df.columns]

fig, axes = plt.subplots(len(target_cols), 1, figsize=(14, 3 * len(target_cols)), constrained_layout=True)
if len(target_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, target_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="#4c72b0")
    ax.set_title(f"Phân phối của {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Số lượng")

plt.show()

## 3. Kết Luận Nhanh

Dùng notebook này để phát hiện sớm:
- Cột thiếu nhiều dữ liệu
- Biến bị lệch phân phối hoặc có outlier
- Mối quan hệ mạnh giữa các chỉ số CAMELS
- Những ngân hàng hoặc năm có dữ liệu bất thường

## 4. Phân Tích Ngoại Lệ

Phần này kiểm tra các giá trị ngoại lệ bằng quy tắc IQR và vẽ boxplot cho các biến CAMELS quan trọng.

In [ ]:
outlier_cols = [
    "total_assets", "total_deposits", "total_loans", "total_equity",
    "npl_amount", "loan_loss_provision", "npl_ratio", "llp_ratio",
    "roa", "roe", "nim", "cir", "eta", "etd", "lta", "ltd", "gta"
]
outlier_cols = [col for col in outlier_cols if col in df.columns]

outlier_rows = []
for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_rows.append({
        "cột": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "ngưỡng_dưới": lower,
        "ngưỡng_trên": upper,
        "số_outlier": int(mask.sum()),
        "tỷ_lệ_outlier_%": round(float(mask.mean() * 100), 2),
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values("số_outlier", ascending=False)
display(outlier_summary)

boxplot_cols = ["npl_ratio", "llp_ratio", "roa", "roe", "nim", "cir", "eta", "etd"]
boxplot_cols = [col for col in boxplot_cols if col in df.columns]

fig, axes = plt.subplots(len(boxplot_cols), 1, figsize=(14, 3 * len(boxplot_cols)), constrained_layout=True)
if len(boxplot_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, boxplot_cols):
    sns.boxplot(x=df[col], ax=ax, color="#8da0cb")
    ax.set_title(f"Boxplot của {col}")
    ax.set_xlabel(col)

plt.show()